In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Path & Data Loading
PROJECT_ROOT = Path(r"C:\Users\USER\OneDrive\New folder\Desktop\KENYA_HOUSEHOLD_ELECTRIC_POWER_CONSUMPTION")
parquet_path = PROJECT_ROOT / "cleaned_household_power.parquet"

df = pd.read_parquet(parquet_path)

if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index)

df = df.sort_index()

# Identify primary active power column safely
power_col = next(
    (col for col in ["Global_active_power", "global_active_power"] if col in df.columns),
    df.columns[0]
)]
df[power_col] = pd.to_numeric(df[power_col], errors="coerce")

# 2. Temporal & Load Regime Features
df["Hour"] = df.index.hour
df["DayOfWeek"] = df.index.dayofweek  # 0=Mon, 6=Sun
df["Weekday_Name"] = df.index.day_name()
df["Month"] = df.index.month
df["Month_Name"] = df.index.strftime("%b")

# Load regimes based on quartiles (or domain thresholds) of active power
df["Load_Regime"] = pd.qcut(
    df[power_col],
    q=4,
    labels=["Q1: Low/Idle", "Q2: Moderate", "Q3: Elevated", "Q4: Peak Draw"]
)

print(f"Loaded shape: {df.shape}")
print(f"Target active power column: '{power_col}'")
df[[power_col, "Hour", "Weekday_Name", "Month_Name", "Load_Regime"]].head()



SyntaxError: unmatched ']' (505602117.py, line 22)

In [2]:
# 1. Conditional summary by Hour of Day
hourly_stats = df.groupby("Hour")[power_col].agg(
    count="count",
    mean="mean",
    median="median",
    std="std",
    iqr=lambda x: x.quantile(0.75) - x.quantile(0.25),
    max_peak="max"
).round(3)

print("=== Top 5 Hours by MEAN Active Power ===")
print(hourly_stats.sort_values("mean", ascending=False).head(5))

print("\n=== Top 5 Hours by PEAK (MAX) Active Power ===")
print(hourly_stats.sort_values("max_peak", ascending=False).head(5))

# 2. Conditional summary by Day of Week (ordered Mon -> Sun)
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
daily_stats = df.groupby("Weekday_Name")[power_col].agg(
    mean="mean",
    median="median",
    std="std",
    var="var",
    iqr=lambda x: x.quantile(0.75) - x.quantile(0.25),
    max_peak="max"
).reindex(day_order).round(3)

daily_stats["CV"] = (daily_stats["std"] / daily_stats["mean"]).round(3)
print("\n=== Daily Variability & Central Tendency ===")
print(daily_stats)
print(f"\nHighest Std Dev Day: {daily_stats['std'].idxmax()} ({daily_stats['std'].max()})")
print(f"Highest IQR Day:     {daily_stats['iqr'].idxmax()} ({daily_stats['iqr'].max()})")

# 3. Conditional summary by Month
monthly_stats = df.groupby(["Month", "Month_Name"])[power_col].agg(
    mean="mean",
    median="median",
    std="std",
    iqr=lambda x: x.quantile(0.75) - x.quantile(0.25),
    max_peak="max"
).round(3)
print("\n=== Monthly Summary ===")
print(monthly_stats)

# 4. Conditional summary by Load Regime
regime_stats = df.groupby("Load_Regime", observed=False)[power_col].agg(
    count="count",
    share_pct=lambda x: len(x) / len(s) * 100,
    mean="mean",
    median="median",
    std="std",
    min="min",
    max="max"
).round(3)
print("\n=== Load Regime Summary ===")
print(regime_stats)

# 5. Sub-metering comparison
sub_cols = [c for c in ["Sub_metering_1", "Sub_metering_2", "Sub_metering_3"] if c in df.columns]
if sub_cols:
    sub_stats = pd.DataFrame({
        "mean": df[sub_cols].mean(),
        "median": df[sub_cols].median(),
        "std": df[sub_cols].std(),
        "iqr": df[sub_cols].quantile(0.75) - df[sub_cols].quantile(0.25),
        "skew": df[sub_cols].skew(),
        "kurtosis": df[sub_cols].kurt(),
        "max": df[sub_cols].max(),
        "zero_share_pct": (df[sub_cols] == 0).mean() * 100
    }).round(3)
    print("\n=== Sub-Metering Comparison ===")
    print(sub_stats)

NameError: name 'df' is not defined

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Path & Data Loading
PROJECT_ROOT = Path(r"C:\Users\USER\OneDrive\New folder\Desktop\KENYA_HOUSEHOLD_ELECTRIC_POWER_CONSUMPTION")
parquet_path = PROJECT_ROOT / "cleaned_household_power.parquet"

df = pd.read_parquet(parquet_path)

if not isinstance(df.index, pd.DatetimeIndex):
    df.index = pd.to_datetime(df.index)

df = df.sort_index()

# Identify primary active power column safely
power_col = next(
    (col for col in ["Global_active_power", "global_active_power"] if col in df.columns),
    df.columns[0]
)
df[power_col] = pd.to_numeric(df[power_col], errors="coerce")

# 2. Temporal & Load Regime Features
df["Hour"] = df.index.hour
df["DayOfWeek"] = df.index.dayofweek  # 0=Mon, 6=Sun
df["Weekday_Name"] = df.index.day_name()
df["Month"] = df.index.month
df["Month_Name"] = df.index.strftime("%b")

# Load regimes based on quartiles of active power
df["Load_Regime"] = pd.qcut(
    df[power_col],
    q=4,
    labels=["Q1: Low/Idle", "Q2: Moderate", "Q3: Elevated", "Q4: Peak Draw"]
)

print(f"Loaded shape: {df.shape}")
print(f"Target active power column: '{power_col}'")
df[[power_col, "Hour", "Weekday_Name", "Month_Name", "Load_Regime"]].head()

Loaded shape: (968294, 18)
Target active power column: 'Global_active_power'


,Global_active_power,Hour,Weekday_Name,Month_Name,Load_Regime
DateTime,,,,,
2006-12-16 17:24:00,4.216,17,Saturday,Dec,Q4: Peak Draw
2006-12-16 17:25:00,5.360,17,Saturday,Dec,Q4: Peak Draw
2006-12-16 17:26:00,5.374,17,Saturday,Dec,Q4: Peak Draw
2006-12-16 17:27:00,5.388,17,Saturday,Dec,Q4: Peak Draw
2006-12-16 17:28:00,3.666,17,Saturday,Dec,Q4: Peak Draw


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")

# --- Chart 1: Histogram + KDE & ECDF ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=200)

sns.histplot(s, bins=60, kde=True, stat="density", ax=axes[0], color="#1f77b4")
axes[0].axvline(mean_val, color="red", linestyle="--", linewidth=1.5, label=f"Mean ({mean_val:.2f})")
axes[0].axvline(median_val, color="green", linestyle="-.", linewidth=1.5, label=f"Median ({median_val:.2f})")
axes[0].set_title("Active Power Density (Histogram + KDE)")
axes[0].set_xlabel(power_col)
axes[0].legend()

sns.ecdfplot(s, ax=axes[1], color="#2ca02c", linewidth=1.8)
axes[1].axhline(0.50, color="gray", linestyle=":", linewidth=1)
axes[1].axhline(0.95, color="orange", linestyle=":", linewidth=1)
axes[1].set_title("Empirical CDF (ECDF) of Active Power")
axes[1].set_xlabel(power_col)
axes[1].set_ylabel("Cumulative Proportion")

plt.tight_layout()
plt.show()

# --- Chart 2: Boxplot & Violin Plot by Day of Week ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5), dpi=200)

sns.boxplot(data=df, x="Weekday_Name", y=power_col, order=day_order, showfliers=False, ax=axes[0])
axes[0].set_title("Active Power by Day of Week (Excl. Extreme Outliers)")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=25)

# Sample up to 100k rows for fast violin rendering
sample_df = df.sample(n=min(100_000, len(df)), random_state=42)
sns.violinplot(data=sample_df, x="Weekday_Name", y=power_col, order=day_order, inner="quartile", cut=0, ax=axes[1])
axes[1].set_title("Violin + Quartiles by Day of Week (Sampled)")
axes[1].set_xlabel("")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.show()

# --- Chart 3: Quantile Chart & Hourly Mean/Peak Bar Chart ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5), dpi=200)

pct_grid = np.linspace(0.01, 0.99, 99)
q_vals = s.quantile(pct_grid)
axes[0].plot(pct_grid * 100, q_vals.values, color="#9467bd", linewidth=2)
axes[0].set_title("Quantile Function (1st to 99th Percentile)")
axes[0].set_xlabel("Percentile")
axes[0].set_ylabel(power_col)

hourly_stats["mean"].plot(kind="bar", ax=axes[1], color="#ff7f0e", alpha=0.85)
axes[1].set_title("Mean Active Power by Hour of Day")
axes[1].set_xlabel("Hour (0–23)")
axes[1].set_ylabel("Mean Power")

plt.tight_layout()
plt.show()